# Demo: Point Process Models

*FW 532 · Python version*

This demo has five parts. First we *simulate* presence points from an inhomogeneous Poisson process
(IPP). Then we go the other way, as in an SDM: we *fit* a point process model to one simulated
pattern, using downweighted Poisson regression. Next we ask how many background (quadrature) points
are enough. The second and third parts follow the supplementary material of Renner et al. (2015,
Appendix S1, Section 5), and connect to Figure 1 of Valavi et al. (2022). Fourth, we use the same
trick to fit flexible models (boosted regression trees and Maxent) as point process models, and
compare them with a linear GLM on a species with strongly nonlinear responses. Finally, we add
sampling bias: we check a model with Ripley's K function, and compare three remedies (a bias
covariate, an area-interaction model, and thinning the records).

You Will Need: numpy, pandas, matplotlib, scipy, statsmodels, scikit-learn, elapid, rasterio, and the course helper module
`sdmlab.py` (in the same folder as this notebook). Run the cells in order from top to bottom.

In [ ]:
# Running on Google Colab? This cell downloads the course files. (Anywhere else it does nothing.)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fw532"):
        !git clone -q --depth 1 https://github.com/Hutchinson-Lab/fw532.git /content/fw532
        %pip install -q rasterio elapid
    os.chdir("/content/fw532/python")

# Can we sample from an IPP in Python to understand it better?

First, let's go back to our simulation of abundance data, to produce an intensity raster. (The R
version of this demo uses the spatstat package to sample point patterns. Python has no direct
equivalent, so we write the sampler ourselves below; it takes only a few lines.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

import sdmlab as sdm   # course helpers for rasters

env_stack = sdm.load_env()

In [ ]:
# scale variables
env_stack = env_stack.scale()

intercept = 2
beta1 = 0.5
beta2 = -0.5
beta3 = 0.25
beta4 = -0.05
beta5 = -0.2
mdr = sdm.scale(env_stack["Mean.Diurnal.Range"])
iso = sdm.scale(env_stack["Isothermality"])
mtwq = sdm.scale(env_stack["Mean.Temperature.of.Wettest.Quarter"])
lambda_map = np.exp(intercept
                    + beta1 * mdr
                    + beta2 * iso
                    + beta3 * mtwq
                    + beta4 * mdr**2          # quadratic term
                    + beta5 * iso * mtwq)     # interaction term

sdm.plot_map(lambda_map, env_stack, title="lambda");

In [ ]:
n_points = 1000
# any number will work as the seed; using the same one will give the same results
sim_pts = env_stack.sample_random(n_points, seed=1234)  # could change this from random to a procedure reflecting sampling bias

# get the mean abundances that correspond to the points we selected:
lambdas = sdm.extract(lambda_map, env_stack, sim_pts["x"], sim_pts["y"])

# sample from these means (with a *different* seed; see Lab 1):
rng = np.random.default_rng(5678)
abund = rng.poisson(lam=lambdas)   # sampling from a Poisson instead of binomial now

# put response and variables together in a DataFrame:
abund_df = sim_pts.copy()
abund_df.insert(0, "abund", abund)

# plot the three relevant climate variables and the mean abundance raster
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), constrained_layout=True)
for ax, name in zip(axes.flat, ["Mean.Diurnal.Range", "Isothermality",
                                "Mean.Temperature.of.Wettest.Quarter"]):
    sdm.plot_map(env_stack[name], env_stack, ax=ax, title=name)
sdm.plot_map(lambda_map, env_stack, ax=axes[1, 1], title="Mean Abundance");

Here λ is the mean count at a *site* (a cell). A point process uses the same kind of surface as an
*intensity*: the expected number of points per unit area. Under an inhomogeneous Poisson process
(IPP), the number of points that land in a cell is Poisson with mean λ × (cell area), so the counts
above are what an IPP would give if we measured area in "cells".

The number of points an IPP generates is Poisson distributed with mean equal to the integral of the
intensity over the whole region, $\Lambda = \int_D \lambda(s)\,ds$. For a raster, the integral is a
sum over cells: $\Lambda = \sum_{\text{cells}} \lambda_{\text{cell}} \times (\text{cell area})$. Our
cells are 1/6 × 1/6 degree, so each has an area of 1/36 ≈ 0.028 square degrees. (`np.nansum()` adds
up the values and skips the `nan` ocean cells.)

In [ ]:
cell_area = env_stack.res**2                 # square degrees per cell
sum_lambda = np.nansum(lambda_map)
print("sum of lambda   :", round(sum_lambda, 1))
print("cell area       :", round(cell_area, 5), "square degrees")
print("expected count  :", round(sum_lambda * cell_area, 1))

So an IPP with this intensity would produce about 22,600 points: enough to crowd the map. Below we
divide the intensity by `scale_factor` to get a sparser pattern.

**How to sample from an IPP on a raster.** Because λ is constant within each cell, an IPP breaks into
independent pieces, one per cell: the number of points in a cell is Poisson with mean
λ_cell × (cell area), and those points are spread uniformly at random within the cell. That's all the
function below does. (spatstat's `rpoispp()` in R gets the same kind of random pattern by a different
route.)

In [ ]:
def sample_ipp(lam, env, rng):
    """Sample points from an inhomogeneous Poisson process whose intensity is the map `lam`
    (expected points per square degree; nan = outside the region)."""
    mean_counts = np.nan_to_num(lam) * env.res**2          # expected number of points in each cell
    counts = rng.poisson(mean_counts)                       # how many points land in each cell
    row, col = np.nonzero(counts)                           # the cells that got at least one point
    n_per_cell = counts[row, col]
    x_centre, y_centre = env.xy(np.repeat(row, n_per_cell), np.repeat(col, n_per_cell))
    n = n_per_cell.sum()
    # place each point uniformly at random within its cell
    x = x_centre + rng.uniform(-0.5, 0.5, size=n) * env.res
    y = y_centre + rng.uniform(-0.5, 0.5, size=n) * env.res
    return pd.DataFrame({"x": x, "y": y})

You can run this cell repeatedly to see samples from the IPP (`default_rng()` with no seed gives new
random numbers each time):

In [ ]:
scale_factor = 25   # Experimenting with scaling. Without dividing, the map is crowded with points.
rng = np.random.default_rng()
ipp_samp = sample_ipp(lambda_map / scale_factor, env_stack, rng)   # sample from the IPP

ax = sdm.plot_map(lambda_map / scale_factor, env_stack, cmap="YlGn",
                  title=f"IPP sample with {len(ipp_samp)} points")   # plot the (scaled) lambda as a base
ax.scatter(ipp_samp["x"], ipp_samp["y"], s=3, color="red");          # add the sampled points on top in red

The number of points that are generated is Poisson distributed with mean equal to the integral of the
intensity over the entire domain. If you sample a few times, you might believe that the numbers are
coming from a Poisson distribution with that mean. Let's check with ten draws (a *list
comprehension*, `[... for i in range(10)]`, repeats the expression ten times and collects the
results in a list):

In [ ]:
expected_count = np.nansum(lambda_map / scale_factor) * cell_area
counts = [len(sample_ipp(lambda_map / scale_factor, env_stack, rng)) for i in range(10)]
print("counts       :", counts)
print("expected     :", round(expected_count, 1))
print("mean of draws:", np.mean(counts), "  sd of draws:", round(np.std(counts, ddof=1), 1),
      "  Poisson sd:", round(np.sqrt(expected_count), 1))

This is why the scaling matters too: dividing λ by `scale_factor` divides the expected count by the
same amount.

> **The units of area matter.** λ is a rate *per unit area*, so its values only mean something once
> the units of area are fixed. If we measured area in pixels (each cell 1 × 1), the expected count
> would simply be the *sum* of the cell values (about 815,000 here, before scaling). In square
> degrees each cell is 1/36, so the same values give 36 times fewer points. A square degree isn't a
> fixed area on the ground, either: it shrinks toward the poles.

We might think of a background sample as coming from a homogeneous Poisson process instead. Sampling
the entire geographic area regardless of the species intensity.

In [ ]:
bkgd_map = np.where(np.isnan(lambda_map), np.nan, 0.3)   # You can play with the constant here to see how it changes the number of points
bkgd_samp = sample_ipp(bkgd_map, env_stack, rng)

ax = sdm.plot_map(bkgd_map, env_stack, cmap="Greys", vmin=0, vmax=1, colorbar=False,
                  title=f"Background sample with {len(bkgd_samp)} points")
ax.scatter(bkgd_samp["x"], bkgd_samp["y"], s=3, color="blue");

land_area = np.sum(~np.isnan(lambda_map)) * cell_area
print("land area:", round(land_area), "square degrees;  expected points:", round(0.3 * land_area))

For a constant intensity, the integral is just the constant times the area of the region. In Maxent
(Lab 6), background points play this role: a sample of the available environments that is used to
approximate the integral of λ over the region (Renner et al. 2015).

# Fitting a point process model to presence-only data

So far we have simulated *from* a point process. An SDM goes the other way: given presence-only
records, estimate the intensity. Let's treat one simulated pattern as our presence records. Because
we know the true λ, we can check the estimates.

In [ ]:
rng = np.random.default_rng(2468)
pres_pts = sample_ipp(lambda_map / scale_factor, env_stack, rng)   # our "presence-only records"
len(pres_pts)

Taking logs of λ (divided by `scale_factor`) gives the true model, a log-linear function of the
covariates:

$$
\ln \lambda(s) = \beta_0 + \beta_1\,\text{MDR} + \beta_2\,\text{Iso} + \beta_3\,\text{MTWQ} + \beta_4\,\text{MDR}^2 + \beta_5\,\text{Iso} \times \text{MTWQ},
$$

with intercept $\beta_0$ = `intercept` − log(`scale_factor`). We'll fit a model of exactly this
form, so any differences from the truth come from estimation, not misspecification.

In [ ]:
true_coefs = pd.Series([intercept - np.log(scale_factor), beta1, beta2, beta3, beta4, beta5],
                       index=["const", "mdr", "iso", "mtwq", "mdr^2", "iso:mtwq"])

def design_matrix(df):
    """Columns of the model: intercept, the three covariates, the quadratic and the interaction."""
    X = df[["mdr", "iso", "mtwq"]].copy()
    X["mdr^2"] = df["mdr"] ** 2
    X["iso:mtwq"] = df["iso"] * df["mtwq"]
    return sm.add_constant(X)

The log-likelihood of an IPP with presence points $s_1, \dots, s_m$ in a region $A$ is

$$
\ell(\beta) = \sum_{i=1}^{m} \ln \lambda(s_i) \;-\; \int_A \lambda(s)\,ds .
$$

The integral has no closed form, so it is approximated by *quadrature*: evaluate λ at a set of
quadrature points and add up the values with weights (areas). Quadrature points are what SDM software
calls **background points**. Seen this way, background points are not pseudo-absences. They are a
device for estimating the integral, so the question to ask about them is how many are needed to
estimate it accurately (Renner et al. 2015). We'll come back to that question below. (In R, the
spatstat function `ppm()` fits these models directly; the R version of this demo compares it with the
approach below.)

## Downweighted Poisson regression (DWPR)

Berman & Turner (1992) showed that the quadrature approximation turns the IPP log-likelihood into a
weighted Poisson regression, so any GLM software can fit it. Renner et al. (2015, Appendix S1,
Section 5.2) gave a simple recipe, *downweighted Poisson regression*. Take the $m$ presence points and
$n$ quadrature points drawn at random from the region. Then:

* give each **quadrature point** weight $w = |A|/n$ (its share of the region's area) and response $z = 0$;
* give each **presence point** a tiny weight $w = \epsilon$ and response $z = 1/\epsilon$;
* fit a Poisson GLM of $z$ with these weights.

The weighted Poisson log-likelihood, $\sum_k w_k (z_k \ln \mu_k - \mu_k)$, is then

$$
\sum_{i=1}^{m} \ln \lambda(s_i) \;-\; \frac{|A|}{n}\sum_{j=1}^{n} \lambda(q_j) \;-\; \underbrace{\epsilon \sum_{i=1}^{m} \lambda(s_i)}_{\approx\, 0},
$$

which is the IPP log-likelihood, with the integral estimated as |A| × (average of λ over the
quadrature points). In statsmodels the weights go in `var_weights=`, which plays the role of the
`weights=` argument of R's `glm()`.

The quadrature points are drawn from the region's grid cells, so first we make a table of every cell
and its covariate values:

In [ ]:
row, col = np.nonzero(~np.isnan(lambda_map))                 # every cell in the region
all_cells = pd.DataFrame({"mdr": mdr[row, col], "iso": iso[row, col], "mtwq": mtwq[row, col]})
region_area = len(all_cells) * cell_area                     # |A|, in square degrees

pres_covs = pd.DataFrame({name: sdm.extract(m, env_stack, pres_pts["x"], pres_pts["y"])
                          for name, m in [("mdr", mdr), ("iso", iso), ("mtwq", mtwq)]})

def dwpr_data(quad_pts, pres_df=None, eps=1e-6):
    """Presences + quadrature points, with DWPR weights (wt) and response (z)."""
    if pres_df is None:
        pres_df = pres_covs       # covariates at the presences of our first species
    dat = pd.concat([pres_df.assign(pres=1), quad_pts[["mdr", "iso", "mtwq"]].assign(pres=0)],
                    ignore_index=True)
    dat["wt"] = np.where(dat["pres"] == 1, eps, region_area / len(quad_pts))   # quadrature weights
    dat["z"] = dat["pres"] / dat["wt"]                                         # 1/eps at presences, 0 elsewhere
    return dat

def fit_dwpr(quad_pts, eps=1e-6):
    """Fit an IPP by downweighted Poisson regression (Renner et al. 2015, Appendix S1, Section 5.2)."""
    dat = dwpr_data(quad_pts, eps=eps)
    wt, z = dat["wt"].to_numpy(), dat["z"].to_numpy()
    fit = sm.GLM(z, design_matrix(dat), family=sm.families.Poisson(), var_weights=wt).fit()
    mu = fit.fittedvalues.to_numpy()
    return {"fit": fit,
            "loglik": np.sum(wt * (z * np.log(mu) - mu)),             # the IPP log-likelihood
            "quad_intensity": mu[dat["pres"].to_numpy() == 0]}        # fitted intensity at quadrature points

Let's fit it with 10,000 random quadrature points, Maxent's default number of background points:

In [ ]:
rng = np.random.default_rng(1357)
quad_10k = all_cells.iloc[rng.choice(len(all_cells), size=10000, replace=False)]
dwpr_10k = fit_dwpr(quad_10k)
pd.DataFrame({"truth": true_coefs, "DWPR": dwpr_10k["fit"].params,
              "SE": dwpr_10k["fit"].bse}).round(3)

The estimates land close to the true coefficients, mostly within about two standard errors. Unlike
presence–background logistic regression, DWPR estimates the intercept on the scale of the intensity
(points per square degree), so its fitted values are a proper intensity map:

In [ ]:
pred = dwpr_10k["fit"].predict(design_matrix(all_cells))
pred_map = np.full(lambda_map.shape, np.nan)
pred_map[row, col] = pred                                    # put the predictions back on the grid

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
vmax = np.nanmax(pred_map)
sdm.plot_map(lambda_map / scale_factor, env_stack, ax=axes[0], title="True intensity", vmin=0, vmax=vmax)
sdm.plot_map(pred_map, env_stack, ax=axes[1], title="DWPR estimate", vmin=0, vmax=vmax);

> **The link to logistic regression and Maxent (IWLR).** Fithian & Hastie (2013) showed that a
> logistic regression of presences against background points, with a huge weight $W$ on the
> background points, recovers the same slopes. This is *infinitely weighted logistic regression*
> (IWLR). It is how `elapid` and R's `maxnet` fit Maxent, with $W = 100$ (Lab 6). The intercept,
> though, is arbitrary: it is shifted by log(W·n/|A|). That's why Maxent's raw output is only
> *proportional* to the intensity (Appendix S1, Section 5.1).

In [ ]:
dat = pd.concat([pres_covs.assign(pres=1), quad_10k.assign(pres=0)], ignore_index=True)
w_iwlr = (1e6) ** (1 - dat["pres"])        # W = 10^6 on background points, 1 on presences
fit_iwlr = sm.GLM(dat["pres"], design_matrix(dat), family=sm.families.Binomial(),
                  var_weights=w_iwlr).fit()
print(pd.DataFrame({"IWLR": fit_iwlr.params, "DWPR": dwpr_10k["fit"].params}).round(3))
print("intercept gap:", round(dwpr_10k["fit"].params["const"] - fit_iwlr.params["const"], 4),
      "   log(W n / |A|):", round(np.log(1e6 * len(quad_10k) / region_area), 4))

> **What the intercept can't tell you: records vs. individuals.** Think of λ as the species'
> abundance (individuals per square degree), with only 1 in `scale_factor` = 25 individuals becoming
> a record. The records then follow an IPP with intensity λ/25, and that is what the model
> estimates. The fitted intercept is close to `intercept` − log(25), and the slopes don't depend on
> `scale_factor` at all. With real data the 25 is unknown. It mixes how abundant the species is with
> how hard people looked, so presence records alone can't tell a rare, well-surveyed species from a
> common, poorly surveyed one. Phillips et al. (2017) call this unknown ratio of true abundance to
> the model's estimate *c*.
>
> That matters as soon as you want a probability of presence. If individuals occur independently,
> the number in a cell of area *a* is Poisson with mean *a*·*c*·λ̂, where λ̂ is the fitted
> intensity. So the probability that the species is present is 1 − exp(−*a*·*c*·λ̂). Without *c*,
> the same formula only gives the probability that a cell holds at least one *record*:

In [ ]:
p_record = 1 - np.exp(-cell_area * pred_map)                   # at least one record in the cell
p_present = 1 - np.exp(-cell_area * scale_factor * pred_map)   # species present: needs c = scale_factor

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
sdm.plot_map(p_record, env_stack, ax=axes[0], title="P(at least one record)", vmin=0, vmax=1)
sdm.plot_map(p_present, env_stack, ax=axes[1], title="P(present), if we knew c = 25", vmin=0, vmax=1);

> This isn't about the units of area: switching from square degrees to km² changes *a* and λ̂ in
> opposite directions and leaves *a*·λ̂ unchanged. Maxent's cloglog output (Lab 6) uses the same
> formula, but replaces the unknown *c*, *a*, and intercept with a constant based on the model's
> entropy. So it is a probability of presence in a quadrat whose size depends on the unknown *c*
> (Phillips et al. 2017).
>
> That's a different problem from the "scale dependence" described by Renner & Warton (2013), where
> Maxent's intercept and predicted probabilities change with the spatial resolution or the number of
> background points. DWPR's area weights fix that problem. No choice of weights fixes this one:
> estimating *c* needs information beyond the presence records, such as presence–absence surveys or
> an estimate of total population size.

# Are there enough background points?

Every number above depends on the integral being estimated well. With too few quadrature points, the
log-likelihood (and so the coefficients, standard errors, and model comparisons) carry extra error
that has nothing to do with the species. Renner et al. (2015) suggest two ways to check.

## Check 1: does the log-likelihood settle down?

Refit the model with more and more random quadrature points and see where the maximized
log-likelihood stops changing (Appendix S1, Section 5.3; Renner et al. 2015, Fig. 2b). Because the
points are random, we repeat each size several times to see how much the answer varies from one set
of quadrature points to another.

We also have a reference value. Our covariates are constant within each grid cell, so using *every*
cell as a quadrature point (weight = one cell's area) computes the integral exactly at the resolution
of our data.

In [ ]:
dwpr_all = fit_dwpr(all_cells)   # every cell: the exact integral at this resolution

n_quad = [500, 1000, 2000, 5000, 10000, 20000, 50000]
n_reps = 20
rng = np.random.default_rng(8642)
log_liks = np.array([[fit_dwpr(all_cells.iloc[rng.choice(len(all_cells), size=n, replace=False)])["loglik"]
                      for n in n_quad]
                     for rep in range(n_reps)])     # n_reps rows x len(n_quad) columns

fig, ax = plt.subplots(figsize=(8, 4.5))
for j, n in enumerate(n_quad):
    ax.scatter(np.full(n_reps, n), log_liks[:, j], s=15, color="black", alpha=0.4)
ax.axhline(dwpr_all["loglik"], color="red", linestyle="--", label="all cells (exact)")
ax.set_xscale("log")
ax.set_xticks(n_quad, [f"{n:,}" for n in n_quad])   # label the sizes we used
ax.minorticks_off()
ax.set_xlabel("Number of random quadrature points (log scale)")
ax.set_ylabel("Maximized log-likelihood")
ax.set_title(f"{n_reps} random sets of quadrature points at each size")
ax.legend();

With a few hundred quadrature points, the log-likelihood can be off by tens of units, depending only
on which points were drawn. The spread narrows as $n$ grows.

This is the same check as Figure 1 of Valavi et al. (2022), the model benchmark we read in Week 6.
Before choosing the number of background points for their study, they refitted MaxEnt with 100 to
100,000 random background points, repeating each size 10 times. They compared the results with a
"gold standard" model that used every cell in the landscape as background, just like our all-cells
reference line. Their yardstick was predictive performance (test AUC) rather than the log-likelihood;
Renner et al. (2015) suggest either. Valavi et al. chose 50,000 background points, the size at which
the replicates came close to the gold standard.

## Check 2: the standard error of the integral

Renner et al. (2015, p. 370) point out that this spread can be predicted with a familiar formula.
With $n$ random quadrature points, the integral is estimated as |A| times the *average* fitted
intensity at those points. That's a sample mean, so its standard error is the usual one:

$$
\text{SE}(n) = \frac{|A|\, s}{\sqrt{n}},
$$

where $s$ is the standard deviation of the fitted intensity across the region, estimated from the
quadrature points of an initial fit. Turned around, this tells us how many points we need. If we want
the SE to be at most some tolerance $\tau$ (in log-likelihood units), we need

$$
n \;\ge\; \left(\frac{|A|\, s}{\tau}\right)^2 .
$$

Renner et al. used $\tau = 2$. That is a sensible scale: differences of a couple of log-likelihood
units are what decide model comparisons. For example, a likelihood-ratio test for one extra term at
the 5% level needs a difference of 1.92.

In [ ]:
s = np.std(dwpr_10k["quad_intensity"], ddof=1)    # spread of the fitted intensity, from the 10,000-point fit

def se_loglik(n):
    return region_area * s / np.sqrt(n)

tolerance = 2
n_required = int(np.ceil((region_area * s / tolerance) ** 2))
print("s                    :", round(s, 3))
print("SE with 10,000 points:", round(se_loglik(10000), 2))
print("n required (SE <= 2) :", n_required)
print("cells in the region  :", len(all_cells))

How well does the formula predict the simulation? Compare the standard deviation of the 20
log-likelihoods at each size with SE(n):

In [ ]:
pd.DataFrame({"simulated SD": log_liks.std(axis=0, ddof=1),
              "formula SE": se_loglik(np.array(n_quad))},
             index=pd.Index(n_quad, name="n")).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for j, n in enumerate(n_quad):
    ax.scatter(np.full(n_reps, n), log_liks[:, j], s=15, color="black", alpha=0.4)
n_grid = np.logspace(np.log10(400), np.log10(60000), 100)
ax.plot(n_grid, dwpr_all["loglik"] + 2 * se_loglik(n_grid), color="blue", label="± 2 SE from the formula")
ax.plot(n_grid, dwpr_all["loglik"] - 2 * se_loglik(n_grid), color="blue")
ax.axhline(dwpr_all["loglik"], color="red", linestyle="--", label="all cells (exact)")
ax.axvline(n_required, color="blue", linestyle=":", label="n needed for SE ≤ 2")
ax.set_ylim(log_liks.min() - 5, log_liks.max() + 5)
ax.set_xscale("log")
ax.set_xticks(n_quad, [f"{n:,}" for n in n_quad])   # label the sizes we used
ax.minorticks_off()
ax.set_xlabel("Number of random quadrature points (log scale)")
ax.set_ylabel("Maximized log-likelihood")
ax.set_title("Simulation vs. the standard-error formula")
ax.legend(loc="lower right");

The formula tracks the simulation reasonably well. (With only 20 replicates, the simulated standard
deviations are themselves noisy, by roughly ±15%.) So one quick initial fit tells us roughly how many
background points are needed, without running all the replicates. For this species the rule's answer
(printed above) is the same order as the 50,000 points that Valavi et al. (2022) settled on by the
slower, replicate-based route. At the largest sizes the simulated
spread is a bit *smaller* than the formula predicts. That's because we are sampling without
replacement from a finite set of cells: with half of all cells in the sample, the SE shrinks by a
factor of about $\sqrt{1 - n/N}$, and with every cell the integral is exact.

## How much does it matter for the coefficients?

The log-likelihood matters for comparing models. What about the coefficients themselves? Here is how
much they vary across random sets of quadrature points, next to their usual (model-based) standard
errors:

In [ ]:
rng = np.random.default_rng(9753)
coef_sd = {"model SE": dwpr_all["fit"].bse}
for n in [1000, 10000, 50000]:
    coefs = pd.DataFrame([fit_dwpr(all_cells.iloc[rng.choice(len(all_cells), size=n, replace=False)])["fit"].params
                          for rep in range(n_reps)])
    coef_sd[f"quadrature SD, n={n}"] = coefs.std(ddof=1)
pd.DataFrame(coef_sd).round(3)

With 1,000 quadrature points, the error from quadrature is comparable to the statistical
uncertainty in the coefficients, and that extra error isn't reflected in the reported standard
errors. With tens of thousands of points it becomes small by comparison. Renner et al. (2015) make the
same point: quadrature error can be modest relative to the coefficients' standard errors, but only
once there are enough points.

> **Questions**
>
> * Maxent's default is 10,000 background points. Is that enough for this species, by the τ = 2 rule?
> * Does the formula for the required number of points depend on how many presences you have? Try
>   `scale_factor = 5` (about five times as many presences) and rerun: how do $s$ and the required
>   $n$ change, and why?
> * Which features of a species' intensity surface would make the integral harder to estimate, so
>   that more background points are needed?

> **Exercise:** Our quadrature points were *random* cells. Try a regular grid instead: use every
> k-th row and column of the grid (e.g. `all_cells` restricted to `(row % k == 0) & (col % k == 0)`,
> with weight |A|/n as before) for k = 2, 4, 8, 16. Compare each log-likelihood with the exact value.
> Do grids reach it faster than random points do? Renner et al. (2015, Fig. 2a) checked convergence
> with a grid of decreasing spacing.

# Beyond GLMs: flexible models as point process models

DWPR turns fitting a point process model into fitting a weighted Poisson regression. So any method
that can fit a Poisson model with observation weights can fit a point process model, not just GLMs.
Renner et al. (2015) make this point for DWPR, and Fithian & Hastie (2013) make the same point for
weighted logistic regression. Two flexible methods from this course qualify:

* **Boosted regression trees.** A Poisson BRT can take exactly the same response `z` and weights `wt`
  as the GLM. The trees learn the shapes of the responses and any interactions from the data, and the
  model still estimates the intensity on its true scale.
* **Maxent.** `elapid` (like R's `maxnet`) fits Maxent by infinitely weighted logistic regression (see
  the IWLR note above). It expands each covariate into many features (linear, quadratic, product,
  hinge) and uses L1 regularization to keep only the ones that help (Lab 6).

Using point-process weights for BRTs is not the usual practice. In their benchmark, Valavi et al.
(2022) fitted BRTs as presence-vs-background *classifiers* (Bernoulli loss), with the background
points down-weighted so that their total weight equals that of the presences. That is the approach
Renner et al. (2015) call statistically naïve. Valavi et al. used point-process weighting only for
GLMs and GAMs.

Our first species is a poor test of flexible methods, though. Its true intensity has a quadratic term
and an interaction, but the GLM we fitted contained *exactly* those terms: it was told the true
formula. Those effects were also mild. In a real analysis nobody tells you the form. So let's simulate
a second species whose responses a simple GLM can't represent, and compare three models that are
*not* told the form: a linear GLM, a Poisson BRT, and Maxent.

## A species with nonlinear responses

Species 2 has a hump-shaped response to mean diurnal range (an optimum near its average), a sharp
threshold in the temperature of the wettest quarter, and a steady decline with isothermality:

$$
\ln \lambda_2(s) = \alpha_2 - 0.8\,\text{MDR}^2 + 1.5 \cdot \mathbb{1}[\text{MTWQ} > 0] - 0.5\,\text{Iso},
$$

where $\mathbb{1}[\cdot]$ is 1 when the condition holds and 0 otherwise. We pick $\alpha_2$ so that
about 900 points are expected.

In [ ]:
def shape2(d):
    """Species 2's log intensity, minus its intercept: hump + step + linear."""
    return -0.8 * d["mdr"] ** 2 + 1.5 * (d["mtwq"] > 0) - 0.5 * d["iso"]

alpha2 = np.log(900 / (np.sum(np.exp(shape2(all_cells))) * cell_area))   # gives ~900 expected points
lambda2_map = np.full(lambda_map.shape, np.nan)
lambda2_map[row, col] = np.exp(alpha2 + shape2(all_cells))             # the true intensity map

rng = np.random.default_rng(4321)
pres_pts2 = sample_ipp(lambda2_map, env_stack, rng)   # training "presence records"
rng = np.random.default_rng(5555)
test_pts2 = sample_ipp(lambda2_map, env_stack, rng)   # an independent test pattern from the same process

def covariates_at(pts):
    """The three covariates at a set of points."""
    return pd.DataFrame({name: sdm.extract(m, env_stack, pts["x"], pts["y"])
                         for name, m in [("mdr", mdr), ("iso", iso), ("mtwq", mtwq)]})

pres_covs2 = covariates_at(pres_pts2)
test_covs2 = covariates_at(test_pts2)
print("training points:", len(pres_covs2), "  test points:", len(test_covs2))

ax = sdm.plot_map(lambda2_map, env_stack, title="True intensity of species 2, with its training points")
ax.scatter(pres_pts2["x"], pres_pts2["y"], s=2, color="red");

## Fitting the three models

All three models get the same 50,000 random quadrature (background) points. That's the number Valavi
et al. (2022) chose, and close to what the standard-error rule asked for with species 1. (Checking it
for species 2 is one of the questions below.)

In [ ]:
rng = np.random.default_rng(6666)
quad2 = all_cells.iloc[rng.choice(len(all_cells), size=50000, replace=False)]
dat2 = dwpr_data(quad2, pres_df=pres_covs2)   # DWPR response (z) and weights (wt) for species 2
covs = ["mdr", "iso", "mtwq"]

**A linear GLM** (DWPR), the simplest model you might try without knowing the form:

In [ ]:
fit_linear = sm.GLM(dat2["z"], sm.add_constant(dat2[covs]), family=sm.families.Poisson(),
                    var_weights=dat2["wt"]).fit()

**A Poisson BRT** with the same response and weights. In R this is the gbm package with
`distribution = "poisson"` and `weights =`. In scikit-learn, `GradientBoostingClassifier` (Lab 5) has
no Poisson loss, but `HistGradientBoostingRegressor` does (`loss="poisson"`), and it accepts weights
through `sample_weight=`. It first bins each covariate into at most 255 values, which makes it fast.
Settings map roughly to gbm's: `learning_rate` is the shrinkage, and `max_leaf_nodes=3` gives
two-split trees like gbm's `interaction.depth = 2`. Instead of cross-validation, we let it stop adding
trees when the loss on a held-out 20% of the data stops improving (*early stopping*).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

fit_brt = HistGradientBoostingRegressor(loss="poisson", learning_rate=0.02, max_leaf_nodes=3,
                                        max_iter=3000, early_stopping=True, validation_fraction=0.2,
                                        n_iter_no_change=50, random_state=0)
fit_brt.fit(dat2[covs], dat2["z"], sample_weight=dat2["wt"])
print("number of trees chosen by early stopping:", fit_brt.n_iter_)

**Maxent** with `elapid`, given the presences and the same quadrature points as background. We use the
same feature classes as maxnet's defaults (for more than 80 presences) and a regularization multiplier
of 1. We also ask for 50 hinge features per covariate, as maxnet uses. elapid's default of 10 is
noticeably less flexible here. `transform="raw"` makes `predict()` return a value proportional to
exp(linear predictor).

In [ ]:
import elapid

fit_maxent = elapid.MaxentModel(feature_types=["linear", "quadratic", "product", "hinge"],
                                beta_multiplier=1.0, n_hinge_features=50, transform="raw")
fit_maxent.fit(dat2[covs], dat2["pres"])

Maxent's output is only *proportional* to the intensity, because the IWLR intercept is arbitrary. For
a Poisson point process, the maximum-likelihood intercept makes the expected number of points,
$\int_A \lambda(s)\,ds$, equal the number observed. So we rescale Maxent's output to make its
integral, estimated with the same quadrature points, equal to the number of presences:

In [ ]:
maxent_scale = len(pres_covs2) / (region_area * fit_maxent.predict(quad2[covs]).mean())

def maxent_intensity(d):
    return maxent_scale * fit_maxent.predict(d[covs])

## Comparing the models

First, the test log-likelihood of each model on the independent test pattern (with the integral
computed exactly over every cell), with the truth and a constant intensity for reference:

In [ ]:
def test_loglik(intensity, test_covs):
    """IPP log-likelihood of the test points, for a function that returns the fitted intensity."""
    return np.sum(np.log(intensity(test_covs))) - np.sum(intensity(all_cells)) * cell_area

pd.Series({
    "truth":      test_loglik(lambda d: np.exp(alpha2 + shape2(d)), test_covs2),
    "linear GLM": test_loglik(lambda d: fit_linear.predict(sm.add_constant(d[covs])), test_covs2),
    "BRT":        test_loglik(lambda d: fit_brt.predict(d[covs]), test_covs2),
    "Maxent":     test_loglik(maxent_intensity, test_covs2),
    "constant":   test_loglik(lambda d: np.full(len(d), len(pres_covs2) / region_area), test_covs2),
}, name="test log-likelihood").round(1)

(A `lambda` is a small unnamed function, so `lambda d: ...` means "a function of `d` that returns
...". Each one here returns a model's fitted intensity for a table of covariates.)

The maps, on a common color scale capped at the true maximum (`imshow` colors higher values with the
top color):

In [ ]:
pred_maps = {"Truth": lambda2_map}
for name, intensity in [("Linear GLM (DWPR)", lambda d: fit_linear.predict(sm.add_constant(d[covs]))),
                        ("BRT (Poisson, DWPR weights)", lambda d: fit_brt.predict(d[covs])),
                        ("Maxent (elapid)", maxent_intensity)]:
    m = np.full(lambda_map.shape, np.nan)
    m[row, col] = intensity(all_cells)
    pred_maps[name] = m

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), constrained_layout=True)
for ax, (name, m) in zip(axes.flat, pred_maps.items()):
    sdm.plot_map(m, env_stack, ax=ax, title=name, vmin=0, vmax=np.nanmax(lambda2_map))

And the response curves: each model's log intensity as one covariate varies, with the other two held at
0 (their average). The models differ in their overall level, so we center each curve (subtract its
mean) to compare *shapes*:

In [ ]:
log_intensity = {
    "truth":      lambda d: alpha2 + shape2(d),
    "linear GLM": lambda d: np.log(fit_linear.predict(sm.add_constant(d[covs], has_constant="add"))),
    "BRT":        lambda d: np.log(fit_brt.predict(d[covs])),
    "Maxent":     lambda d: np.log(fit_maxent.predict(d[covs])),
}
styles = {"truth": ("black", "--"), "linear GLM": ("orange", "-"), "BRT": ("forestgreen", "-"),
          "Maxent": ("purple", "-")}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, v in zip(axes, ["mdr", "mtwq", "iso"]):
    sweep = pd.DataFrame({"mdr": np.zeros(200), "iso": 0.0, "mtwq": 0.0})   # others held at 0
    sweep[v] = np.linspace(all_cells[v].quantile(0.01), all_cells[v].quantile(0.99), 200)
    for name, f in log_intensity.items():
        curve = np.asarray(f(sweep), dtype=float)
        color, linestyle = styles[name]
        ax.plot(sweep[v], curve - curve.mean(), color=color, linestyle=linestyle, lw=2, label=name)
    ax.set_xlabel(v)
    ax.set_ylabel("log intensity (centered)")
    ax.set_title(v)
axes[0].legend();

The linear GLM can't bend: it draws a nearly flat line through the hump and a tilted line through the
step. Its test log-likelihood is far below the truth's, though still well above a constant intensity.
Both flexible models recover the shapes and come much closer to the truth, and close to each other.
Their curves show their characters. Maxent's quadratic features capture the hump smoothly, and its
hinges trace the step. The BRT captures the step crisply but draws everything as small stairs, and it
goes flat beyond the data. This isn't a contest with a fixed winner: a different species, or more
careful tuning of either model, could reverse their order.

> **Questions**
>
> * Why do the BRT and Maxent need just as many quadrature points as the GLM? (What do the quadrature
>   points approximate?)
> * Apply the standard-error rule (Check 2) to species 2, using the fitted BRT or Maxent intensity at
>   the quadrature points to estimate $s$. Are 50,000 points enough?
> * Look at the response curves near the edges of the data. Which model would you trust to predict
>   into new conditions, such as a future climate?
> * Tune the BRT (`max_leaf_nodes`, `learning_rate`) and Maxent (`beta_multiplier`,
>   `feature_types`, `n_hinge_features`) as in Labs 5 and 6. To keep the test pattern honest, tune on
>   a separate *validation* pattern (simulate one!). Does the order change?

# Sampling bias and spatial dependence

So far every individual had the same chance, 1 in `scale_factor`, of becoming a record. Real records
come from people, and people look much harder in some places than others: near towns, roads, and
field stations. In point-process terms, records are a *thinned* version of the individuals. If an
individual at location $s$ becomes a record with probability $p(s)$, independently of the others,
the records are again a Poisson process, with intensity

$$
\lambda_{\text{records}}(s) = p(s)\,\lambda(s), \qquad \ln \lambda_{\text{records}}(s) = \ln \lambda(s) + \ln p(s).
$$

When $p$ is constant, a PPM recovers λ up to a constant, which only shifts the intercept. When
$p(s)$ varies, the model estimates λ × effort. Because the bias enters the log intensity as an extra
term, $\ln p(s)$, any environmental covariate that is correlated with effort absorbs part of it.
Then the slopes are biased, not just the intercept.

This section follows Renner et al. (2015) and its Appendix S1 (Section 3). We simulate biased
records for species 1, fit a PPM that ignores the bias, and check it with Ripley's K function. Then
we try three remedies: measuring the bias, modeling the extra spatial dependence, and thinning the
records as the R package spThin does. (The R version of this demo uses spatstat and spThin for these
steps. Python has no direct equivalents among our packages, so we write short versions ourselves.)

In [ ]:
from scipy.spatial import cKDTree                 # fast nearest-neighbour searches
from scipy.ndimage import distance_transform_edt   # distance of each cell to the coast
from sklearn.neighbors import BallTree             # great-circle distances

## Simulating biased records

We place 30 "towns" at random, favoring places with a warm wettest quarter (high MTWQ), and let
survey effort fall off with the distance $d$ to the nearest town:

$$
p(s) = \frac{1}{1 + d(s)/(50\ \text{km})} .
$$

Effort is 1 at a town, 1/2 at 50 km, and about 1/11 at 500 km. We scale the intensity so that about
1,000 records are expected, as before.

In [ ]:
cell_x, cell_y = env_stack.xy(row, col)    # centres of all the cells in the region
rng = np.random.default_rng(97531)
town_prob = np.exp(all_cells["mtwq"].to_numpy())             # towns favor high MTWQ
town_cells = rng.choice(len(all_cells), size=30, replace=False, p=town_prob / town_prob.sum())
towns = pd.DataFrame({"x": cell_x[town_cells], "y": cell_y[town_cells]})

# great-circle distance (km) from every cell to the nearest town
tree = BallTree(np.radians(towns[["y", "x"]].to_numpy()), metric="haversine")
d, _ = tree.query(np.radians(np.column_stack([cell_y, cell_x])), k=1)
dist_km = np.full(lambda_map.shape, np.nan)
dist_km[row, col] = 6371.0 * d[:, 0]
effort_map = 1 / (1 + dist_km / 50)        # relative chance that an individual becomes a record

bias_scale = np.nansum(lambda_map * effort_map) * cell_area / 1000   # about 1,000 records expected
rng = np.random.default_rng(8642)
bias_pts = sample_ipp(lambda_map * effort_map / bias_scale, env_stack, rng)   # biased records
len(bias_pts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), constrained_layout=True)
sdm.plot_map(effort_map, env_stack, ax=axes[0], title="Survey effort, with towns")
axes[0].scatter(towns["x"], towns["y"], marker="^", s=12, color="red")
sdm.plot_map(lambda_map / scale_factor, env_stack, ax=axes[1], title="True intensity, with biased records")
axes[1].scatter(bias_pts["x"], bias_pts["y"], s=1, color="black");

The records pile up around the towns. Far from any town the species is just as common, but hardly
anyone looks.

## A PPM that ignores the bias

First, the analysis we would do if we didn't think about effort: the same PPM as before, with only
the environmental covariates. Below is a DWPR function for this section that also accepts extra
covariates (we'll need them for the remedies), and a table of every cell's covariates, including the
bias covariate for Remedy 1. Effort changes over tens of kilometers near the towns, so we use 20,000
quadrature points.

In [ ]:
log_dist_map = np.log1p(dist_km / 50)     # the bias covariate, ln(1 + d/50 km), for Remedy 1
cov_maps = {"mdr": mdr, "iso": iso, "mtwq": mtwq, "log_dist": log_dist_map}

def covariates_at(pts):
    """Covariate values (including the bias covariate) at a set of points."""
    return pd.DataFrame({name: sdm.extract(m, env_stack, pts["x"], pts["y"]) for name, m in cov_maps.items()})

cells_b = pd.DataFrame({"x": cell_x, "y": cell_y, **{name: m[row, col] for name, m in cov_maps.items()}})
rng = np.random.default_rng(1357)
quad_b = cells_b.iloc[rng.choice(len(cells_b), size=20000, replace=False)].reset_index(drop=True)

def fit_ppm(pres_df, extra=(), quad_df=None, eps=1e-6):
    """DWPR fit with the environmental terms plus any `extra` covariates (columns of pres_df)."""
    quad_df = quad_b if quad_df is None else quad_df
    cols = ["mdr", "iso", "mtwq", *extra]
    dat = pd.concat([pres_df[cols].assign(pres=1), quad_df[cols].assign(pres=0)], ignore_index=True)
    wt = np.where(dat["pres"] == 1, eps, region_area / len(quad_df))
    z = (dat["pres"] / wt).to_numpy()
    X = design_matrix(dat)
    for name in extra:
        X[name] = dat[name]
    fit = sm.GLM(z, X, family=sm.families.Poisson(), var_weights=wt).fit()
    mu = fit.fittedvalues.to_numpy()
    return {"fit": fit,
            "log_pl": np.sum(wt * (z * np.log(mu) - mu)),   # log (pseudo)likelihood
            "lam_pres": mu[:len(pres_df)]}                  # fitted intensity at the presences

def coef_table(**fits):
    """The environmental coefficients of several fits, next to the truth."""
    out = pd.DataFrame({"truth": true_coefs})
    for name, f in fits.items():
        out[name] = f["fit"].params[true_coefs.index]
    return out.round(3)

fit_unbiased = fit_ppm(covariates_at(pres_pts))   # the unbiased records from earlier
fit_ignoring = fit_ppm(covariates_at(bias_pts))
coef_table(unbiased=fit_unbiased, ignoring_bias=fit_ignoring).assign(SE=fit_ignoring["fit"].bse.round(3))

The MTWQ slope is about twice its true value, and the MDR slope is too large as well. The model has
learned the towns' preference for a warm wettest quarter as if it were the species'. Nothing in the
output warns us: the standard errors are small, and the estimates look as plausible as the unbiased
ones.

## Ripley's K: is there clustering the covariates don't explain?

A Poisson PPM assumes the records are independent once the covariates are taken into account
(Renner et al. 2015). Ripley's K function checks this. $K(r)$ is the expected number of *other*
points within distance $r$ of a typical point, divided by the intensity; for a homogeneous Poisson
process it equals $\pi r^2$. The *inhomogeneous* K function (Baddeley, Møller & Waagepetersen 2000)
weights each pair of points by $1/(\lambda(s_i)\lambda(s_j))$, using the fitted intensity, so that it
still equals $\pi r^2$ when the fitted Poisson model is right. It is easiest to read as $L(r) - r$,
where $L(r) = \sqrt{K(r)/\pi}$:

* $L(r) - r \approx 0$: no more clustering than the covariates explain;
* $L(r) - r > 0$: extra clustering at distance $r$;
* $L(r) - r < 0$: points are more evenly spaced (*regular*) than the model predicts.

Points near the coast have part of their neighbourhood in the ocean, where there can't be any
points. The simplest fix, the *border correction*, only counts pairs around points that are at least
$r$ from the edge of the region. Distances are in degrees: 1° of latitude is about 111 km.

In [ ]:
land = ~np.isnan(lambda_map)
# distance (degrees) from each cell to the nearest cell outside the region
edge_dist_map = (distance_transform_edt(np.pad(land, 1))[1:-1, 1:-1] - 0.5) * env_stack.res
r_grid = np.arange(0.05, 5.0001, 0.05)    # distances (degrees) at which to compute L(r)

def l_inhom(pts, lam, r=r_grid):
    """Inhomogeneous L function with the border correction (like spatstat's Linhom)."""
    xy = pts[["x", "y"]].to_numpy()
    b = sdm.extract(edge_dist_map, env_stack, xy[:, 0], xy[:, 1])     # each point's distance to the edge
    pairs = cKDTree(xy).query_pairs(r.max(), output_type="ndarray")   # all pairs closer than max(r)
    i = np.concatenate([pairs[:, 0], pairs[:, 1]])                    # count each pair both ways
    j = np.concatenate([pairs[:, 1], pairs[:, 0]])
    dij = np.linalg.norm(xy[i] - xy[j], axis=1)
    w = 1 / (lam[i] * lam[j])
    num = np.array([w[(dij <= rk) & (b[i] >= rk)].sum() for rk in r])
    den = np.array([(1 / lam[b >= rk]).sum() for rk in r])
    return np.sqrt(num / den / np.pi)

How far from 0 is too far? We simulate 39 patterns from the fitted model, refit the model to each
one, and compute the same function for each. The grey band, the *envelope*, shows the range of the 39
simulated curves at each distance. A single test over all distances is the Diggle–Cressie–
Loosmore–Ford (DCLF) test (Baddeley et al. 2014). It adds up the squared deviation of the observed
curve from the simulations' average, and compares that total with the same quantity for each
simulation. With 39 simulations the smallest possible p-value is 1/40 = 0.025. (Renner et al. used
1,001 simulations.)

In [ ]:
def k_test(pts, extra=(), nsim=39, seed=1):
    """Monte Carlo test of a Poisson PPM with the inhomogeneous L function, plus the DCLF test."""
    fit = fit_ppm(covariates_at(pts), extra)
    L_obs = l_inhom(pts, fit["lam_pres"])
    X = design_matrix(cells_b)                       # the fitted intensity map, for simulating
    for name in extra:
        X[name] = cells_b[name]
    lam_map = np.full(lambda_map.shape, np.nan)
    lam_map[row, col] = fit["fit"].predict(X)
    rng = np.random.default_rng(seed)
    sims = []
    for k in range(nsim):
        sim = sample_ipp(lam_map, env_stack, rng)              # simulate from the fitted model
        f = fit_ppm(covariates_at(sim), extra)                 # refit to the simulated pattern
        sims.append(l_inhom(sim, f["lam_pres"]))
    sims = np.array(sims)
    # DCLF: squared deviation from the mean of the simulations (leave-one-out for each simulation)
    t_obs = np.sum((L_obs - sims.mean(axis=0))**2)
    loo_mean = (sims.sum(axis=0) - sims) / (nsim - 1)
    t_sim = np.sum((sims - loo_mean)**2, axis=1)
    return {"L": L_obs, "lo": sims.min(axis=0), "hi": sims.max(axis=0), "mean": sims.mean(axis=0),
            "p": (1 + np.sum(t_sim >= t_obs)) / (nsim + 1)}

def plot_k(ax, k, title):
    ax.fill_between(r_grid, k["lo"] - r_grid, k["hi"] - r_grid, color="0.8")
    ax.plot(r_grid, k["mean"] - r_grid, "r--", lw=1)
    ax.plot(r_grid, k["L"] - r_grid, "k", lw=1.5)
    ax.set(xlabel="r (degrees)", ylabel="L(r) − r", title=f"{title} (DCLF p = {k['p']:.3f})")

k_unbiased = k_test(pres_pts, seed=1)
k_ignoring = k_test(bias_pts, seed=2)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
plot_k(axes[0], k_unbiased, "Unbiased records")
plot_k(axes[1], k_ignoring, "Biased records");

The unbiased records stay inside the envelope. The biased records rise far above it from about 1°
(roughly 100 km) onward. They are much more clustered than the environment explains. The test can't
say *why*. Extra clustering can come from the species (dispersal, social groups), from an
environmental variable missing from the model, or from the observers. Here we know it's the
observers, but in a real analysis that's a judgment call.

## Remedy 1: measure the bias

If we can measure what drives effort, we can put it in the model. Renner et al. (2015) used the
distances to the nearest main road and urban area. Here the bias covariate is
$\ln(1 + d/50\ \text{km})$. That is exactly $-\ln p(s)$, so its true coefficient is $-1$. Then,
following Renner et al., we predict with the bias covariate set to a common level, $d = 0$: the
intensity of records if everywhere were surveyed as hard as next to a town.

In [ ]:
fit_bias = fit_ppm(covariates_at(bias_pts), extra=("log_dist",))
print(coef_table(ignoring_bias=fit_ignoring, with_bias_covariate=fit_bias))
print("bias coefficient:", round(fit_bias["fit"].params["log_dist"], 3), "  (true value: -1)")

The environmental slopes are back near the truth, and the bias coefficient is close to $-1$. The K
test agrees that nothing important is missing now:

In [ ]:
k_bias = k_test(bias_pts, extra=("log_dist",), seed=3)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
plot_k(ax, k_bias, "Biased records, with the bias covariate");

To compare the maps, we use each model's *environmental* part only, $\exp(\mathbf{x}(s)'\boldsymbol\beta)$.
That is the same as setting the bias covariate to 0 (that is, $d = 0$). We divide each map by its
mean, because the intercepts mean different things in different models and only the *relative*
intensity is comparable:

In [ ]:
def env_part_map(fit):
    """exp(environmental part of the log intensity), as a map."""
    X = design_matrix(cells_b)
    m = np.full(lambda_map.shape, np.nan)
    m[row, col] = np.exp(X.to_numpy() @ fit.params[X.columns].to_numpy())
    return m

relative = lambda m: m / np.nanmean(m)
rel_maps = {"Truth": relative(lambda_map),
            "Ignoring the bias": relative(env_part_map(fit_ignoring["fit"])),
            "Bias covariate, at d = 0": relative(env_part_map(fit_bias["fit"]))}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4), constrained_layout=True)
for ax, (title, m) in zip(axes, rel_maps.items()):
    sdm.plot_map(m, env_stack, ax=ax, title=title, vmin=0, vmax=3)

Ignoring the bias exaggerates the intensity across the central and southern United States and
northern Mexico, where most of the towns are, and understates it in the north. With the bias
covariate, the map matches the truth closely.

## Remedy 2: model the extra spatial dependence

Often we can't measure effort. Another option is to let the model describe the clustering it can't
explain. Renner et al. (2015) did this with an *area-interaction* model, a Gibbs point process. Its
intensity at a location $s$ also depends on the other records nearby:

$$
\ln \lambda(s \mid \text{records}) = \mathbf{x}(s)'\boldsymbol\beta + \theta\, t(s),
$$

where $t(s)$ is the fraction of a disc of radius $r$ around $s$ that is also within $r$ of another
record, i.e., covered by the discs around the other records. $\theta > 0$ means records are more
likely near other records (clustering), and $\theta < 0$ means they avoid each other (inhibition).

Gibbs models are fitted by *maximum pseudolikelihood*. For this model that means the same weighted
Poisson regression as DWPR, with $t(s)$ as one more covariate, evaluated at the presences and the
quadrature points (Baddeley & Turner 2000; Renner et al. 2015). In R, spatstat fits this model with
`ppm(..., interaction = AreaInter(r))`, as in Appendix S1, Section 3.6.

In [ ]:
# t(s): the fraction of the disc of radius r around each location that lies within r of another
# record. We approximate the disc by 177 evenly spaced points and check each one for a record within r.
disc = np.array([(a, b) for a in np.linspace(-1, 1, 15) for b in np.linspace(-1, 1, 15) if a*a + b*b <= 1])

def covered_fraction(pts, recs, r, own_record=False):
    xy, rec_xy = pts[["x", "y"]].to_numpy(), recs[["x", "y"]].to_numpy()
    disc_pts = (xy[:, None, :] + r * disc[None, :, :]).reshape(-1, 2)   # disc points around every location
    owner = np.repeat(np.arange(len(xy)), len(disc))                    # which location each belongs to
    dist, idx = cKDTree(rec_xy).query(disc_pts, k=2)                    # the two nearest records
    if own_record:   # for a record, ignore the record itself
        dist = np.where(idx[:, 0] == owner, dist[:, 1], dist[:, 0])
    else:
        dist = dist[:, 0]
    return (dist <= r).reshape(len(xy), len(disc)).mean(axis=1)

def fit_area_inter(r):
    pres_df = covariates_at(bias_pts).assign(covered=covered_fraction(bias_pts, bias_pts, r, own_record=True))
    quad_df = quad_b.assign(covered=covered_fraction(quad_b, bias_pts, r))
    return fit_ppm(pres_df, extra=("covered",), quad_df=quad_df)

The radius $r$ has to be chosen. As in Appendix S1 (spatstat's `profilepl()`), we fit the model for
several radii and keep the one with the largest pseudolikelihood:

In [ ]:
radii = [0.25, 0.5, 1, 1.5, 2]   # degrees
ai_fits = [fit_area_inter(r) for r in radii]
log_pl = [f["log_pl"] for f in ai_fits]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(radii, log_pl, "o-")
ax.set(xlabel="Disc radius r (degrees)", ylabel="Log pseudolikelihood")
plt.show()

fit_ai = ai_fits[int(np.argmax(log_pl))]
print("best radius:", radii[int(np.argmax(log_pl))], "  theta:", round(fit_ai["fit"].params["covered"], 2))
coef_table(ignoring_bias=fit_ignoring, area_interaction=fit_ai)

The interaction is clearly positive: the model has found the clustering. The MTWQ slope moves part
of the way back toward the truth, because some of the clustering around towns is now explained by
"records near other records" instead of by the climate there. But the other slopes don't simply
improve. Several shrink toward 0, and isothermality's moves *away* from the truth. The interaction
term discounts every cluster, whatever its cause, so it also discounts the species' real hotspots.
And the model doesn't know where the towns are, or that remote areas were barely surveyed. Renner et
al. (2015) add a caution: because the parameters are not maximum-likelihood estimates, the usual
standard errors, likelihood-ratio tests, and AIC may not apply.

## Remedy 3: thin the records, as spThin does

The most common fix in practice works on the data rather than the model: *spatial thinning*. The R
package spThin (Aiello-Lammens et al. 2015) repeatedly removes one of the records that has the most
neighbors closer than a chosen distance, until no two records are that close. The removals are
random, so it runs several times and we keep the run that keeps the most records. The function below
follows the same algorithm. We thin to 100 km, both the biased records and, for comparison, the
unbiased ones:

In [ ]:
def thin_records(pts, km, reps=10, seed=1):
    """Spatial thinning with the spThin algorithm: no two records closer than `km`."""
    lat, lon = np.radians(pts["y"].to_numpy()), np.radians(pts["x"].to_numpy())
    # great-circle distances (km) between all pairs of records
    a = (np.sin((lat[:, None] - lat[None, :]) / 2)**2
         + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin((lon[:, None] - lon[None, :]) / 2)**2)
    too_close = 2 * 6371.0 * np.arcsin(np.sqrt(a)) < km
    np.fill_diagonal(too_close, False)
    rng = np.random.default_rng(seed)
    best = None
    for rep in range(reps):
        keep = np.ones(len(pts), dtype=bool)
        n_close = too_close.sum(axis=1).astype(float)    # number of too-close neighbors of each record
        while n_close.max() > 0:
            i = rng.choice(np.flatnonzero(n_close == n_close.max()))   # one with the most neighbors
            keep[i] = False
            n_close -= too_close[:, i]                   # its neighbors each lose one neighbor
            n_close[i] = -np.inf                         # never pick it again
        if best is None or keep.sum() > best.sum():
            best = keep
    return pts[best].reset_index(drop=True)

thin_biased = thin_records(bias_pts, km=100)
thin_unbiased = thin_records(pres_pts, km=100)
print("biased:  ", len(bias_pts), "->", len(thin_biased))
print("unbiased:", len(pres_pts), "->", len(thin_unbiased))

In [ ]:
ax = sdm.plot_map(effort_map, env_stack, title="Biased records: kept (blue) and removed (red) by thinning to 100 km")
ax.scatter(bias_pts["x"], bias_pts["y"], s=1.5, color="red")
ax.scatter(thin_biased["x"], thin_biased["y"], s=2, color="blue");

Now fit the same PPM to the thinned records, and run the K test:

In [ ]:
fit_thin_biased = fit_ppm(covariates_at(thin_biased))
fit_thin_unbiased = fit_ppm(covariates_at(thin_unbiased))
coef_table(unbiased=fit_unbiased, thinned_unbiased=fit_thin_unbiased,
           ignoring_bias=fit_ignoring, thinned_biased=fit_thin_biased)

In [ ]:
k_thin_biased = k_test(thin_biased, seed=4)
k_thin_unbiased = k_test(thin_unbiased, seed=5)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
plot_k(axes[0], k_thin_unbiased, "Unbiased, thinned")
plot_k(axes[1], k_thin_biased, "Biased, thinned");

Thinning has real benefits and real costs.

**Pros**

* It needs no information about effort, just the records.
* It removed many of the records that crowd around towns, so the MTWQ slope moved toward the truth,
  though not all the way.
* The large-scale clustering in the K plot is much reduced.

**Cons**

* It discarded about half the records, including ones that were perfectly good.
* It flattens the response. Records are thinned hardest wherever they are dense, and that includes
  the species' real hotspots, not just the towns. So most of the other slopes shrink toward 0, and
  isothermality's moves away from the truth.
* Applied to the *unbiased* records, where there was nothing to fix, it made things worse: most
  slopes shrink toward 0 there too.
* The thinned records are no longer a Poisson process. No two are closer than 100 km, so
  $L(r) - r$ plunges below the envelope at short distances. That is regularity created by the
  thinning, not by the species, and it violates the PPM's assumptions in the opposite direction.
* The intercept no longer means anything: the number of records left depends on the thinning
  distance, not on the species.
* The thinning distance is arbitrary, and the results depend on it.

## Comparing the remedies

In [ ]:
coef_table(unbiased=fit_unbiased, ignoring_bias=fit_ignoring, bias_covariate=fit_bias,
           area_interaction=fit_ai, thinned=fit_thin_biased).drop(index="const")   # slopes only

When the source of the bias can be measured, putting it in the model and predicting at a common
level of effort works best. It uses all the records and recovers the species' response. Modeling
the dependence and thinning both reduce the bias in the slope that effort was confounded with
(MTWQ), but neither removes it. Both also flatten the rest of the response, because neither can tell
a cluster of observers from a cluster of the species. The K test is useful either way: it tells us
*that* a Poisson model is missing something, even though it can't tell us what.

> **Questions**
>
> * The towns were placed preferentially in warm wettest quarters. What would happen to the PPM that
>   ignores the bias if the towns were placed at random instead (drop `p=` in `rng.choice()`)? Would
>   the K test still detect clustering?
> * Why does thinning the *unbiased* records flatten the fitted response? Think about which records
>   the algorithm removes first.
> * The area-interaction model treats clustering as if records attracted each other. When would that
>   be biologically reasonable for the *species* rather than the observers?
> * In a real analysis, the bias covariate is never exactly $-\ln p(s)$. What might go wrong if it is
>   only a rough proxy for effort, such as distance to the nearest road?

> **Exercise.** Rerun the thinning with `km=25` and `km=250`. How do the number of records kept, the
> slopes, and the K plot change? Is there a distance that removes the clustering around towns without
> creating regularity, and without flattening the response?

# References

Aiello-Lammens, M. E., Boria, R. A., Radosavljevic, A., Vilela, B., & Anderson, R. P. (2015). spThin:
An R package for spatial thinning of species occurrence records for use in ecological niche models.
*Ecography*, 38(5), 541–545. <https://doi.org/10.1111/ecog.01132>

Baddeley, A., Diggle, P. J., Hardegen, A., Lawrence, T., Milne, R. K., & Nair, G. (2014). On tests of
spatial pattern based on simulation envelopes. *Ecological Monographs*, 84(3), 477–489.
<https://doi.org/10.1890/13-2042.1>

Baddeley, A. J., Møller, J., & Waagepetersen, R. (2000). Non- and semi-parametric estimation of
interaction in inhomogeneous point patterns. *Statistica Neerlandica*, 54(3), 329–350.
<https://doi.org/10.1111/1467-9574.00144>

Baddeley, A., & Turner, R. (2000). Practical maximum pseudolikelihood for spatial point patterns.
*Australian & New Zealand Journal of Statistics*, 42(3), 283–322.
<https://doi.org/10.1111/1467-842X.00128>

Berman, M., & Turner, T. R. (1992). Approximating point process likelihoods with GLIM. *Journal of
the Royal Statistical Society, Series C (Applied Statistics)*, 41(1), 31–38.

Elith, J., Leathwick, J. R., & Hastie, T. (2008). A working guide to boosted regression trees.
*Journal of Animal Ecology*, 77(4), 802–813. <https://doi.org/10.1111/j.1365-2656.2008.01390.x>

Fithian, W., & Hastie, T. (2013). Finite-sample equivalence in statistical models for presence-only
data. *The Annals of Applied Statistics*, 7(4), 1917–1939. <https://doi.org/10.1214/13-AOAS667>

Phillips, S. J., Anderson, R. P., Dudík, M., Schapire, R. E., & Blair, M. E. (2017). Opening the
black box: An open-source release of Maxent. *Ecography*, 40(7), 887–893.
<https://doi.org/10.1111/ecog.03049>

Renner, I. W., & Warton, D. I. (2013). Equivalence of MAXENT and Poisson point process models for
species distribution modeling in ecology. *Biometrics*, 69(1), 274–281.
<https://doi.org/10.1111/j.1541-0420.2012.01824.x>

Renner, I. W., Elith, J., Baddeley, A., Fithian, W., Hastie, T., Phillips, S. J., Popovic, G., &
Warton, D. I. (2015). Point process models for presence-only analysis. *Methods in Ecology and
Evolution*, 6(4), 366–379, including Appendix S1. <https://doi.org/10.1111/2041-210X.12352>

Valavi, R., Guillera-Arroita, G., Lahoz-Monfort, J. J., & Elith, J. (2022). Predictive performance of
presence-only species distribution models: A benchmark study with reproducible code. *Ecological
Monographs*, 92(1), e01486. <https://doi.org/10.1002/ecm.1486>